# ROL 1: Disenador de Datos - Modelado y Generacion
## Caso SUNBURST - Analisis de Gestion de Datos

**Universidad de San Buenaventura** | Gestion de Datos | 3er Semestre

---

### Objetivo
Disenar el modelo de datos del incidente SUNBURST y generar datos sinteticos base usando Python.

### Contenido del Notebook
1. Contexto del Caso SUNBURST
2. Modelo Entidad-Relacion (ER)
3. Generacion de Datos Sinteticos (con anomalias intencionales)
4. Vista Previa de DataFrames
5. Exportacion a CSV
6. Resumen de Anomalias Inyectadas
7. Conclusiones y Decisiones de Diseno

---
## 1. Contexto del Caso SUNBURST

El **ataque SUNBURST** fue uno de los ciberataques mas sofisticados de la historia. En diciembre de 2020, se descubrio que actores maliciosos habian comprometido el sistema de compilacion de **SolarWinds**, insertando malware en la **Plataforma Orion**.

### Datos Clave del Caso
| Aspecto | Detalle |
|---------|--------|
| **Software afectado** | SolarWinds Orion Platform |
| **Versiones comprometidas** | 2019.4 (hasta HF4), 2020.2 y 2020.2 HF1 |
| **Versiones limpias** | 2019.4 HF5, 2020.2.1 |
| **Descargas afectadas** | ~18,000 (estimacion inicial) |
| **Clientes realmente comprometidos** | <100 |
| **Costo para SolarWinds** | USD 52.6 millones |

### Timeline del Ataque
- **Ene 2019**: Primera evidencia de actividad del actor de amenazas
- **Feb 2020**: Se compila e implementa SUNBURST
- **Mar 2020**: Hotfix 5 disponible (parche limpio)
- **Dic 2020**: Se notifica a SolarWinds sobre SUNBURST
- **May 2021**: Investigaciones forenses completas

---
## 2. Modelo Entidad-Relacion (ER)

Se disenaron **4 entidades** que representan los componentes clave del escenario SUNBURST:

```
+----------------------------+        +-----------------------------+
|        CLIENTES            |        |    VERSIONES_SOFTWARE       |
+----------------------------+        +-----------------------------+
| [PK] cliente_id      INT   |        | [PK] version_id       INT  |
|      nombre_org      VARCHAR|        |      nombre_version   VARCHAR|
|      tipo_org        VARCHAR|        |      fecha_release    DATE |
|      pais            VARCHAR|        |      contiene_sunburst BOOL|
|      sector          VARCHAR|        |      fecha_compilacion DATE|
|      criticidad      VARCHAR|        +------------+----------------+
+-----------+----------------+                     |
            |                                      |
            | 1:N                             1:N  |
            |                                      |
            v                                      v
+------------------------------------------------------+
|                 INSTALACIONES                        |
+------------------------------------------------------+
| [PK] instalacion_id         INT                      |
| [FK] cliente_id             INT -> CLIENTES          |
| [FK] version_id             INT -> VERSIONES         |
|      fecha_instalacion      DATE                     |
|      nivel_datos_sensibles  VARCHAR                  |
+--------------------------+---------------------------+
                           |
                      1:N  |
                           v
+------------------------------------------------------+
|              EVENTOS_SEGURIDAD (Rol 2)               |
+------------------------------------------------------+
| [PK] evento_id              INT                      |
| [FK] instalacion_id         INT -> INSTALACIONES     |
|      timestamp              DATETIME                 |
|      tipo_evento            VARCHAR                  |
|      severidad              VARCHAR                  |
|      es_anomalo             BOOLEAN                  |
+------------------------------------------------------+
```

> **Nota**: El diagrama ER tambien se genera como imagen PNG en `visualizations/ER.png`

### Relaciones
- **Clientes -> Instalaciones** (1:N): Un cliente puede tener multiples instalaciones
- **Versiones -> Instalaciones** (1:N): Una version puede estar instalada en multiples organizaciones
- **Instalaciones -> Eventos** (1:N): Una instalacion puede generar multiples eventos de seguridad

---
## 3. Generacion de Datos Sinteticos

### IMPORTANTE: Anomalias intencionales

Los datos generados incluyen **anomalias intencionales** para que el Rol 2 pueda detectarlas y realizar un analisis de calidad realista. En un escenario real, los datos rara vez son perfectos.

### Configuracion Inicial

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import os
from datetime import datetime, timedelta

np.random.seed(42)
fake = Faker('es_ES')
Faker.seed(42)

print('Librerias cargadas correctamente')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

In [ ]:
# Constantes del caso
TIPOS_ORGANIZACION = ['Gobierno Federal', 'Gobierno Estatal', 'Empresa Privada',
                      'Institucion Educativa', 'Organizacion de Salud', 'ONG']
SECTORES = ['Tecnologia', 'Defensa', 'Energia', 'Finanzas',
            'Telecomunicaciones', 'Salud', 'Gobierno', 'Educacion']
CRITICIDADES = ['Alta', 'Media', 'Baja']
NIVELES_DATOS_SENSIBLES = ['Bajo', 'Medio', 'Alto', 'Critico']
PAISES = ['Estados Unidos'] * 7 + ['Reino Unido', 'Canada', 'Alemania', 'Israel',
          'Australia', 'Francia', 'Japon']

### 3.1 Generacion de Clientes (50 registros + anomalias)

In [ ]:
# Importamos la funcion desde el script limpio
import sys
sys.path.insert(0, '../scripts')
from rol1_generacion_datos import generar_clientes, generar_versiones, generar_instalaciones

clientes_df = generar_clientes(n=50)
print(f'\nDimensiones: {clientes_df.shape}')
print(f'\nAnomaliasm detectables:')
print(f'  Nulos en nombre_organizacion: {clientes_df["nombre_organizacion"].isnull().sum()}')
print(f'  Pais vacio: {(clientes_df["pais"].fillna("").str.strip() == "").sum()}')
print(f'  tipo_org invalido: {clientes_df[~clientes_df["tipo_org"].isin(TIPOS_ORGANIZACION)].shape[0]}')
print(f'  criticidad nula: {clientes_df["criticidad"].isnull().sum()}')
print(f'  IDs duplicados: {clientes_df["cliente_id"].duplicated().sum()}')

### 3.2 Generacion de Versiones (8 registros + 1 anomalia)

In [ ]:
versiones_df = generar_versiones()
print(f'\nDimensiones: {versiones_df.shape}')
print(f'Con SUNBURST: {versiones_df["contiene_sunburst"].sum()}')
print(f'fecha_release nula: {versiones_df["fecha_release"].isnull().sum()}')

### 3.3 Generacion de Instalaciones (100 registros + anomalias)

In [ ]:
instalaciones_df = generar_instalaciones(clientes_df, versiones_df, n=100)
print(f'\nDimensiones: {instalaciones_df.shape}')

# Verificar anomalias
cli_ids = set(clientes_df['cliente_id'].unique())
ver_ids = set(versiones_df['version_id'])
print(f'FK cliente_id invalidos: {instalaciones_df[~instalaciones_df["cliente_id"].isin(cli_ids)].shape[0]}')
print(f'FK version_id invalidos: {instalaciones_df[~instalaciones_df["version_id"].isin(ver_ids)].shape[0]}')
print(f'nivel_datos_sensibles nulos: {instalaciones_df["nivel_datos_sensibles"].isnull().sum()}')

---
## 4. Vista Previa de DataFrames

In [ ]:
print('=' * 70)
print('DataFrame: CLIENTES')
print('=' * 70)
print(f'Dimensiones: {clientes_df.shape[0]} filas x {clientes_df.shape[1]} columnas')
clientes_df.head(10)

In [ ]:
clientes_df.info()
print()
clientes_df.describe(include='all')

In [ ]:
print('=' * 70)
print('DataFrame: VERSIONES_SOFTWARE')
print('=' * 70)
versiones_df

In [ ]:
print('=' * 70)
print('DataFrame: INSTALACIONES')
print('=' * 70)
print(f'Dimensiones: {instalaciones_df.shape[0]} filas x {instalaciones_df.shape[1]} columnas')
instalaciones_df.head(10)

In [ ]:
instalaciones_df.info()
print()
instalaciones_df.describe(include='all')

---
## 5. Exportacion a CSV

In [ ]:
os.makedirs('../data', exist_ok=True)

clientes_df.to_csv('../data/clientes.csv', index=False, encoding='utf-8')
versiones_df.to_csv('../data/versiones_software.csv', index=False, encoding='utf-8')
instalaciones_df.to_csv('../data/instalaciones.csv', index=False, encoding='utf-8')

print('Archivos CSV guardados:')
for archivo in ['clientes.csv', 'versiones_software.csv', 'instalaciones.csv']:
    ruta = os.path.join('../data', archivo)
    df_temp = pd.read_csv(ruta)
    print(f'  {archivo}: {len(df_temp)} registros, {os.path.getsize(ruta):,} bytes')

---
## 6. Resumen de Anomalias Inyectadas

Los datos contienen las siguientes anomalias intencionales para evaluacion por el Rol 2:

### Tabla Clientes
| Anomalia | Descripcion | Filas |
|----------|------------|-------|
| Nulos en nombre_organizacion | 3 registros sin nombre | 7, 23, 41 |
| Pais vacio | 2 registros con string vacio | 15, 33 |
| tipo_org invalido | Valor 'Desconocido' | 28 |
| criticidad nula | 1 registro sin criticidad | 45 |
| cliente_id duplicado | ID 12 aparece 2 veces | 12, 51 |

### Tabla Versiones
| Anomalia | Descripcion | Filas |
|----------|------------|-------|
| fecha_release nula | HF2 sin fecha de release | 3 |

### Tabla Instalaciones
| Anomalia | Descripcion | Filas |
|----------|------------|-------|
| Fechas anteriores al release | 3 inconsistencias temporales | 5, 20, 56 |
| FK cliente_id inexistentes | IDs 999 y 888 | 11, 68 |
| FK version_id inexistente | ID 99 | 36 |
| nivel_datos_sensibles nulos | 2 registros | 9, 73 |
| nivel_datos_sensibles invalido | Valor 'Desconocido' | 51 |
| Fecha fuera de rango | Ano 2023 | 89 |

---
## 7. Conclusiones y Decisiones de Diseno

### Decisiones de Modelado
1. **Modelo de 3+1 entidades**: Clientes, Versiones_Software, Instalaciones (Rol 1) + Eventos_Seguridad (Rol 2)
2. **Datos con anomalias**: Se inyectaron anomalias intencionales para simular la realidad de datos imperfectos
3. **Coherencia con el caso real**: Versiones, fechas y distribuciones basadas en el caso Harvard
4. **Reproducibilidad**: seed(42) garantiza resultados identicos en cada ejecucion

### Por que anomalias intencionales?
En un entorno real, los datos **nunca** son perfectos. El valor del Rol 2 (Analista de Calidad) radica precisamente en su capacidad de detectar y documentar problemas como:
- Claves foraneas huerfanas por datos migrados incorrectamente
- Campos nulos por formularios incompletos
- Valores fuera de catalogo por falta de validacion en la entrada
- Inconsistencias temporales por errores humanos

Un analisis de calidad que reporta 100% en todo no es creible ni util.